## Model

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [30]:
class AlexNet(nn.Module):
    def __init__(self):
        super(AlexNet, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channel=3,
            out_channel=96,
            kernel_size=11,
            stride=4,
            padding=0
        )
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2)
        self.conv2 = nn.Conv2d(
            in_channel=96,
            out_channel=256,
            kernel_size=5,
            stride=1,
            padding=2
        )
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2)
        self.conv3 = nn.Conv2d(
            in_channel=256,
            out_channel=384,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.conv4 = nn.Conv2d(
            in_channel=384,
            out_channel=384,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.conv5 = nn.Conv2d(
            in_channel=384,
            out_channel=256,
            kernel_size=3,
            stride=1,
            padding=1
        )
        self.pool3 = nn.MaxPool2d(kernel_size=3, stride=2)

        self.fc1 = nn.Linear(
            in_features=9216,
            out_features=4096
        )
        self.dropout1 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(
            in_features=4096,
            out_features=4096
        )
        self.dropout2 = nn.Dropout(0.5)
        self.fc3 = nn.Linear(
            in_features=4096,
            out_features=1000
        )

    def forward(self, image):
        bs, c, h, w = image.size()
        x = F.relu(self.conv1(image))
        x = self.poo1(x)
        x = F.relu(self.conv2(x))
        x = self.poo2(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = x.view(bs, -1)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = self.dropout2(x)
        x = F.relu(self.fc3(x))

        x = torch.softmax(x, axis=1)
        return x

## Dataset

In [31]:
import torch
import numpy as np
from PIL import Image
from PIL import ImageFile

In [32]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [55]:
class ClassificationDataset:
    def __init__(
        self,
        image_paths,
        targets,
        resize=None,
        augmentations=None
    ):
        self.image_paths = image_paths
        self.targets = targets
        self.resize = resize
        self.augmentations = augmentations

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, item):
        image = Image.open(self.image_paths[item])
        image = image.convert("RGB")
        targets = self.targets[item]

        if self.resize is not None:
            image = image.resize(
                (self.resize[1], self.resize[0]),
                resample=Image.BILINEAR
            )

        image = np.array(image)

        if self.augmentations is not None:
            augmented = self.augmentations(image=image)
            image = augmented["image"]

        # Convert the shape from (height, width, channel) to (channel, height, width) which is expected by PyTorch
        image = np.transpose(image, (2, 0, 1)).astype(np.float32)

        return {
            "image": torch.tensor(image, dtype=torch.float),
            "targets": torch.tensor(targets, dtype=torch.long)
        }

## Engine

In [34]:
import torch
import torch.nn as nn

In [62]:
def train(data_loader, model, optimizer, device):
    model.train()

    for data in data_loader:
        inputs = data["image"]
        targets = data["targets"]

        inputs = inputs.to(device, dtype=torch.float)
        targets = targets.to(device, dtype=torch.float)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = nn.BCEWithLogitsLoss()(outputs, targets.view(-1, 1))
        loss.backward()
        optimizer.step()

In [63]:
def evaluate(data_loader, model, device):
    model.eval()

    final_targets = []
    final_outputs = []

    with torch.no_grad():
        for data in data_loader:
            inputs = data["image"]
            targets = data["targets"]

            inputs = inputs.to(device, dtype=torch.float)
            targets = targets.to(device, dtype=torch.float)

            output = model(inputs)

            targets = targets.detach().cpu().numpy().tolist()
            output = output.detach().cpu().numpy().tolist()

            final_targets.extend(targets)
            final_outputs.extend(output)

    return final_targets, final_outputs

## Model

In [37]:
import torch.nn as nn
import pretrainedmodels

In [73]:
def get_model(pretrained):
    if pretrained:
        model = pretrainedmodels.__dict__["alexnet"](pretrained="imagenet")
    else:
        model = pretrainedmodels.__dict__["alexnet"](pretrained=None)
    model.last_linear = nn.Sequential(
        nn.BatchNorm1d(4096),
        nn.Dropout(p=0.25),
        nn.Linear(in_features=4096, out_features=2048),
        nn.ReLU(),
        nn.BatchNorm1d(2048, eps=1e-05, momentum=0.1),
        nn.Dropout(p=0.5),
        nn.Linear(in_features=2048, out_features=1)
    )
    return model

In [46]:
get_model(True)

/Users/atsushi-takahiro@cookpad.com/hobby/kaggle_notebook/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/atsushi-takahiro@cookpad.com/hobby/kaggle_notebook/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


AlexNet(
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (_features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (dropout0): Dropout(p=0.5, inplace=False)
  (linear0): Linear(in_features=9216, out_features=4096, bias=True)
  (relu0): ReLU(inp

## Train

In [40]:
import os
import pandas as pd
import numpy as np
import albumentations
import torch
from sklearn import metrics
from sklearn.model_selection import train_test_split

In [41]:
# GPU が利用可能か確認
if torch.cuda.is_available():
    print('GPU が利用可能です')
else:
    print('GPU は利用できません。CPU を使用します')

GPU は利用できません。CPU を使用します


In [42]:
# MPS (Metal Performance Shaders) が利用可能か確認
if torch.backends.mps.is_available():
    print("MPS デバイスが利用可能です")
else:
    print("MPS デバイスは利用できません。CPU を使用します")

MPS デバイスが利用可能です


In [67]:
def main():
    data_path = '../../datasets/rsna-pneumonia-detection-challenge/'
    device = "mps"
    epochs = 10

    df = pd.read_csv(os.path.join(data_path, "train.csv"))

    images = df.patientId.values.tolist()
    images = [
        os.path.join(data_path, "train_png", i+"png") for i in images
    ]

    targets = df.Target.values

    model = get_model(pretrained=True)
    model.to(device)

    # TODO: 何をしているのか調べる -> https://stackoverflow.com/questions/58151507/why-pytorch-officially-use-mean-0-485-0-456-0-406-and-std-0-229-0-224-0-2
    mean = (0.485, 0.456, 0.406)
    std = (0.229, 0.224, 0.225)

    aug = albumentations.Compose(
        [
            albumentations.Normalize(
                mean, std, max_pixel_value=255.0
            )
        ]
    )

    # TODO: k-fold交差検証の代わりにホールドアウト検証を使うのはなぜ？
    train_images, valid_images, train_targets, valid_targets = train_test_split(
        images, targets, stratify=targets, random_state=42
    )

    train_dataset = ClassificationDataset(
        image_paths=train_images,
        targets=train_targets,
        resize=(227, 227),
        augmentations=aug
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=16, shuffle=True, num_workers=4
    )

    valid_dataset = ClassificationDataset(
        image_paths=valid_images,
        targets=valid_targets,
        resize=(227, 227),
        augmentations=aug
    )

    valid_loader = torch.utils.data.DataLoader(
        # valid_dataset, batch_size=16, shuffle=True, num_workers=4
        valid_dataset, batch_size=16, shuffle=True, num_workers=0
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

    for epoch in range(epochs):
        train(train_loader, model, optimizer, device=device)
        predictions, valid_targets = evaluate(valid_loader, model, device=device)
        roc_auc= metrics.roc_auc_score(valid_targets, predictions)
        print(
            f"Epoch={epoch}, Valid ROC AUC={roc_auc}"
        )

In [74]:
main()

/Users/atsushi-takahiro@cookpad.com/hobby/kaggle_notebook/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/atsushi-takahiro@cookpad.com/hobby/kaggle_notebook/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/atsushi-takahiro@cookpad.com/.pyenv/versions/3.12.9/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/atsushi-takahiro@cookpad.com/.pyenv/versions/3.12.9/lib/python3.12/multiprocess

KeyboardInterrupt: 